# ZenGuard UEBA Anomaly Detection

This notebook implements the User and Entity Behavior Analytics (UEBA) component of the ZenGuard Framework using the Isolation Forest algorithm.

## System Integration Data Flow

### 1. What Output Comes from the SIEM?
- **Alerts and Correlated Events:** SIEM acts as the central monitoring hub. It aggregates logs from firewalls, IDS, Identity Providers (IdP), and Endpoint Detection and Response (EDR).
- **Parsed Metadata:** Source and destination IPs, timestamps, activity types (e.g., login attempts, external connections), and user/device identifiers.
- **Event Triggers:** SIEM forwards flagged events to the Listener module for UEBA analysis.

### 2. What Input UEBA Needs?
UEBA requires normalized behavioral session telemetry to establish baselines and detect deviations. Features include:
- `session_duration` (numeric)
- `failed_logins` (count)
- `access_time` (hour)
- `device_trust_score` (normalized)
- `privilege_change_attempted` (binary)
- `external_connection` (binary)
- `MFA_bypassed` (binary)

*Note: In this implementation, we use the CICIDS2017 dataset which simulates network flow characteristics to demonstrate anomaly detection as requested.*

### 3. What Output UEBA Produces?
- **Anomaly Risk Score:** A discrete risk score. A score below the anomaly threshold (anomaly metric isolation) generates a high risk score (i.e. 95). Normal sessions receive a lower risk score (i.e. 50).
- **Contextual Flag:** Identified malicious activities vs. benign operations.

### 4. What Inputs SOAR Needs?
- **Source and Destination IPs**
- **UEBA Risk Score** (e.g. 95)
- **Trigger Action Metadata:** Context such as whether to enforce MFA, block IP via Firewall API, or isolate an endpoint.


In [1]:
!pip install kagglehub pandas numpy scikit-learn joblib matplotlib seaborn

In [2]:
import kagglehub
import pandas as pd
import numpy as np
import os
import glob

# Stage 1: Data Ingestion
print('Downloading dataset...')
path = kagglehub.dataset_download("asthana12/cicids2017")
print("Path to dataset files:", path)

# Find the CSV file in the downloaded path
csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
if csv_files:
    data_path = csv_files[0]
    print(f"Loading {data_path}...")
    # Load a sample to keep memory usage low for demonstration
    df = pd.read_csv(data_path, nrows=100000)
    print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
else:
    print("No CSV files found in the dataset path.")


100%|██████████| 230M/230M [00:02<00:00, 110MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/asthana12/cicids2017/versions/1
Loading /root/.cache/kagglehub/datasets/asthana12/cicids2017/versions/1/Friday-WorkingHours-Afternoon-PortScan.csv...
Loaded 100000 rows and 79 columns.


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Stage 2: Data Preprocessing
print("Cleaning columns...")
df.columns = df.columns.str.strip()

# Drop rows with infinite values and NAs
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Select numerical features for Isolation Forest
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
if 'Label' in numeric_features:
    numeric_features.remove('Label') # Avoid target leakage

print(f"Selected {len(numeric_features)} numerical features for anomaly detection.")

X = df[numeric_features]

# Create Preprocessing Pipeline following ml-pipeline-workflow principles
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


Cleaning columns...
Selected 78 numerical features for anomaly detection.


In [4]:
from sklearn.ensemble import IsolationForest

# Stage 3: Model Training (UEBA Isolation Forest)
print("Building Isolation Forest Model Pipeline...")
ueba_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', IsolationForest(n_estimators=100, contamination=0.05, random_state=42, n_jobs=-1))
])

print("Training the UEBA Anomaly Detection Model...")
ueba_model.fit(X)
print("Model training complete.")


Building Isolation Forest Model Pipeline...
Training the UEBA Anomaly Detection Model...
Model training complete.


In [5]:
import joblib

# Stage 4: Model Evaluation and Risk Scoring
print("Generating anomaly scores calculations...")
# -1 for outliers, 1 for inliers
predictions = ueba_model.predict(X)
anomaly_scores_raw = ueba_model.decision_function(X)

# Map to Risk Scores: Anomaly (-1) -> 95 Risk, Normal (1) -> 50 Risk (Referenced in Paper)
risk_scores = np.where(predictions == -1, 95, 50)
df['UEBA_Risk_Score'] = risk_scores
df['UEBA_Anomaly_Score'] = anomaly_scores_raw

high_risk_interactions = df[df['UEBA_Risk_Score'] == 95]
print(f"Detected {len(high_risk_interactions)} anomalous (high-risk) events out of {len(df)} total.")

# Display a sample of anomalous activities
display(high_risk_interactions.head())

# Stage 5: Model Saving for Production Deployment
model_filename = 'zenguard_ueba_model.pkl'
joblib.dump(ueba_model, model_filename)
print(f"Model successfully saved to {model_filename} for SOAR/SIEM integration.")


Generating anomaly scores calculations...
Detected 4997 anomalous (high-risk) events out of 99935 total.


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,UEBA_Risk_Score,UEBA_Anomaly_Score
12,52320,2,2,0,2071,0,2065,6,1035.500000,1455.932862,...,0.000,0,0,0.0,0.000,0,0,BENIGN,95,-0.042747
13,52320,27701,15,6,18467,24,2920,0,1231.133333,1054.492489,...,0.000,0,0,0.0,0.000,0,0,BENIGN,95,-0.023635
28,0,118012435,141,0,0,0,0,0,0.000000,0.000000,...,4036105.291,9795862,66,9420803.7,5219271.767,20000000,5046092,BENIGN,95,-0.000022
31,389,55068559,24,13,3558,7472,1304,0,148.250000,369.486630,...,0.000,20222,20222,55000000.0,0.000,55000000,55000000,BENIGN,95,-0.026679
32,3268,55091905,24,13,3204,7030,1305,0,133.500000,364.357565,...,0.000,24456,24456,55100000.0,0.000,55100000,55100000,BENIGN,95,-0.023272


Model successfully saved to zenguard_ueba_model.pkl for SOAR/SIEM integration.
